In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from pathlib import Path
from copy import deepcopy

from openquake.hazardlib.imt import PGA, SA, RSD595, AvgSA, IMT

from pickagm.distributions import ensemble_ks_bounds

from phd_project.config.config import load_config 
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    ESHM20SiteRupCtxBuilder,
    create_gmm_map,
    create_corr_model_map,
    calculate_gcim_distributions_for_sites,
    get_ground_motion_ensembles_for_sites,
    get_best_ensemble_in_list,
    total_ks_statistic_and_failing_im_penalty,
    optimise_ground_motion_ensembles_for_sites,
    optimise_ground_motion_ensembles_for_sites_with_shuffles,
)
import phd_project.scripts.WP1_ground_motion_set.manage_flatfiles as mf
from phd_project.plotting.plotting import custom_log_formatter

cfg = load_config()

C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\gm_selection.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# AvgSA([0, 6])

In [3]:
# Load the disaggregation data
fp = cfg["proc_data"]["site_hazard"] / "AvgSA_06_disagg_data_60sites.pickle"
with open(fp, "rb") as f:
    disagg_data = pickle.load(f)

fp = cfg["proc_data"]["site_hazard"] / "AvgSA_06_disagg_stats_60sites.pickle"
with open(fp, "rb") as f:
    disagg_stats = pickle.load(f)

# load the site file
sites = pd.read_csv(cfg["hazard_models"]["eshm20_AvgSA_site_model_all"])

# load the flatfiles
flatfile_folder = cfg["proc_data"]["corr_model"] / "reverse" / "flatfiles"
flatfiles = {}
for f in [f for f in os.listdir(flatfile_folder) if f.endswith(".csv")]:
    tag = f.split("_")[0]
    flatfiles[tag] = pd.read_csv(flatfile_folder / f, delimiter=";", index_col=0, low_memory=False)

flatfiles["volcanic"] = pd.read_csv(cfg["raw_data"]["gm_flatfiles"] / "volcanic_lanzanoluzi_flatfile.csv", 
                                    delimiter=";", index_col=0)

# load the preprocessed gm database
gm_database = pd.read_csv(cfg["proc_data"]["gm_database"], sep=",", low_memory=False, header=[0, 1])

In [4]:
# create the average depth map for each TRT
average_depths = {
    "Craton": flatfiles["asc"]["ev_depth_km"].mean(),
    "Non-Subduction Deep": flatfiles["vran"]["ev_depth_km"].mean(),
    "Shallow Default": flatfiles["asc"]["ev_depth_km"].mean(),
    "Subduction Inslab": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Subduction Interface": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Volcanic": flatfiles["volcanic"]["ev_depth_km"].mean(),
}

# the map of what sim trts are allowed to match with what record trts
OK_TRT_MATCHES = {
    "Craton": ["Shallow Default"],
    "Non-Subduction Deep": ["Non-Subduction Deep", "Subduction Inslab", "Subduction"],
    "Shallow Default": ["Shallow Default"],
    "Subduction Inslab": ["Non-Subduction Deep", "Subduction Inslab", "Subduction"],
    "Subduction Interface": ["Subduction Interface"],
    "Volcanic": ["Shallow Default"],
}

occurence = True    # the record selection should be performed based on occurence

In [5]:
# set some parameters for the selection
t_lower = 0.025     # lower SA period considered in selection
t_upper = 6         # upper SA period considered in selection
n_periods = 20      # number of periods to consider in selection

conditioning_imt: IMT = AvgSA([0,6])                      
nonSA_imts: list[IMT] = [AvgSA([0,6]), RSD595(), PGA()] 
sa_periods = np.round(np.geomspace(t_lower, t_upper, num=n_periods), 3)
SA_imts: list[IMT] = [SA(period) for period in sa_periods]
selection_imts: list[IMT] = nonSA_imts[1:] + SA_imts    # not AvgSA but the others (PGA and RSD595)
nonSA_imt_strs: list[str] = [im.string for im in nonSA_imts] # strings match the correlation matrix

# weights of the IMs
weight_rsd595 = 0.25
n_other_ims = len([imt for imt in selection_imts if imt.name == "SA" or imt.name == "PGA"])
imt_weights = np.array([(1-weight_rsd595) / n_other_ims if imt.name != "RSD595" 
                        else weight_rsd595 for imt in selection_imts])
imt_weights /= imt_weights.sum()

# some other things
disagg_type = "TRT_Mag_Dist_Eps"
percentiles = [0.05, 0.16, 0.33, 0.5, 0.67, 0.84, 0.95]     # percentiles of the gcim distribution to return  
assumed_rake = -90                                # assumed rake for RSD595 calculation

In [6]:
# filter the gm_database so that only the selection and conditioning ims are present
gm_db = gm_database.copy()
updated_ims = mf.filter_gm_database_on_imts(
    gm_db["ims"], selection_imts + [conditioning_imt])
updated_ims.columns = pd.MultiIndex.from_product([['ims'], updated_ims.columns])
gm_db = pd.concat([gm_db.drop('ims', axis=1, level=0), updated_ims], axis=1)

with open(r"C:\Users\clemettn\Documents\gm_db.pickle", "wb") as file:
    pickle.dump(gm_db, file)

# Create the GMM Map for AvgSA by reading the logic tree
AvgSA_06_lt_fp = cfg["hazard_models"]["eshm20_AvgSA"] / "gmpe_logic_tree_AvgSA_0to6_median_branch.xml"
gmm_map = create_gmm_map(AvgSA_06_lt_fp)

# get the correlation model map
corr_map = create_corr_model_map(nonSA_imt_strs, sa_periods)

# organise the disagg data
site_poe_disaggs = {}
for (s, r) in disagg_data.keys():
    for site in disagg_data[(s,r)].keys():
        for poe in disagg_data[(s,r)][site][conditioning_imt.name].keys():
            site_poe_disaggs[(site, poe)] = disagg_data[(s,r)][site][conditioning_imt.name][poe]
site_poes = sorted(list(site_poe_disaggs.keys()), key=lambda x: x[0])

In [7]:
# set up the selection context:
basic_selection_ctx = {
    "n_ensembles": 20,
    "n_samples": 30,
    "conditioning_imt": conditioning_imt ,
    "disagg_imt": conditioning_imt.name , # this only works for AvgSA. otherwise used .string 
    "selection_imts": selection_imts ,
    "imt_weights": imt_weights ,
    "sites": sites ,
    "ctx_builder": ESHM20SiteRupCtxBuilder ,
    "ctx_builder_params": ["average_depths", "assumed_rake"] ,
    "average_depths": average_depths ,
    "assumed_rake": assumed_rake ,
    "gmm_map": gmm_map ,
    "corr_map": corr_map ,
    "m_bound_model": "tarbali_and_bradley_2016" ,
    "d_bound_model": "tarbali_and_bradley_2016" ,
    "vs30_bound_model": "tarbali_and_bradley_2016" ,
    "sf_bounds": (0.25, 4),
    "usable_T": t_upper ,
    "max_n_recs": 5 ,
    "p_value": 0.05 ,
    "ok_trt_matches": OK_TRT_MATCHES ,
    "occurence": True ,
}
rng_seed = 1
LOAD_GCIM_IF_EXISTS = True
LOAD_RS_IF_EXISTS = True
LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS = True

## GCIM Distributions

In [8]:
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / f"gcim_dist_AvgSA_06_rake{int(assumed_rake)}.pickle"

if LOAD_GCIM_IF_EXISTS: # load the gcim distributions instead of 
    no_file = False
    if gcim_dist_fp.is_file():
        with open(gcim_dist_fp, "rb") as file:
            gcim_dists = pickle.load(file)
        print("Existing GCIM distribution data loaded...")
    else:
        no_file = True
        print("No existing GCIM distribution data found...")

if (not LOAD_GCIM_IF_EXISTS) or no_file:
    # calculate the gcims and save them
    gcim_dists = calculate_gcim_distributions_for_sites(
        site_poe_disaggs, disagg_stats, conditioning_imt, 
        selection_imts, sites, gmm_map, corr_map, 
        average_depths, assumed_rake, occurence, percentiles)

    with open(gcim_dist_fp, "wb") as file:
        pickle.dump(gcim_dists, file)

Existing GCIM distribution data loaded...


## Preliminary Selection

In [9]:
only_select = []

preliminary_selection_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_prelim_selection_rake{int(assumed_rake)}.pickle"

if LOAD_RS_IF_EXISTS: # load the preliminary record selection instead of calculating 
    no_file = False
    if preliminary_selection_fp.is_file():
        with open(preliminary_selection_fp, "rb") as file:
            preliminary_ensembles = pickle.load(file)
        print("Existing record selection data loaded...")
    else:
        no_file = True
        print("No existing record selection data found...")
        
if (not LOAD_RS_IF_EXISTS) or no_file:
    # do the record selection for all sites and save the results
    candidate_ensembles = get_ground_motion_ensembles_for_sites(
        site_poe_disaggs, disagg_stats, gcim_dists, 
        gm_db, basic_selection_ctx, sites, rng_seed, only_select)
    
    # get the best of the candidate ensembles and save
    obj_func = total_ks_statistic_and_failing_im_penalty

    preliminary_ensembles = {}

    
    for (site, poe), ensembles in candidate_ensembles.items():

        if ensembles == []:
            # no ensemble was found
            preliminary_ensembles[(site, poe)] = None
            continue

        target_cdfs = gcim_dists[(site, poe)]["cdfs"]
        ks_bounds = ensemble_ks_bounds(
            target_cdfs, 
            basic_selection_ctx["n_samples"],
            basic_selection_ctx["p_value"])
        
        obj_func_kwargs = {
            "target_cdfs_list": [np.column_stack([np.log(ksb[1:,0]), ksb[1:,2]]) 
                                for ksb in ks_bounds.values()],
            "upper_ks_bounds_list": [np.column_stack([np.log(ksb[1:,0]), ksb[1:,3]]) 
                                    for ksb in ks_bounds.values()],
            "lower_ks_bounds_list": [np.column_stack([np.log(ksb[1:,0]), ksb[1:,1]]) 
                                    for ksb in ks_bounds.values()],
            "n_recs": basic_selection_ctx["n_samples"],
            "penalty_constant": 10
        }

        e = get_best_ensemble_in_list(
            ensembles, basic_selection_ctx["conditioning_imt"].string, 
            obj_func, obj_func_kwargs)
        preliminary_ensembles[(site, poe)] = e

    # Save the results
    with open(preliminary_selection_fp, "wb") as file:
        pickle.dump(preliminary_ensembles, file)

# Check if all the sites and poes found a set of suitable ground motions
prelim_ensembles_not_passing = []
no_preliminary_ensembles = []
for (site, poe), ensemble in preliminary_ensembles.items():
    
    if ensemble is None:
        prelim_ensembles_not_passing.append((site, poe))
        no_preliminary_ensembles.append((site, poe))
    
    elif not ensemble["ks_passed"]:
        prelim_ensembles_not_passing.append((site, poe))

if len(prelim_ensembles_not_passing) == 0:
    print(f"OK! - Preliminary ensembles pass for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - Preliminary ensembles do not pass for {len(prelim_ensembles_not_passing)} combinations of site and poe")

if no_preliminary_ensembles:
    print(f"No preliminary ensembles found for {len(no_preliminary_ensembles)} combinations of site and poe")

No existing record selection data found...


Selecting GM Ensembles:   0%|          | 0/360 [00:00<?, ?it/s]

Sub-Optimal! - Preliminary ensembles do not pass for 117 combinations of site and poe
No preliminary ensembles found for 26 combinations of site and poe


In [10]:
no_preliminary_ensembles

[(30, np.float64(0.000201)),
 (30, np.float64(0.0001)),
 (36, np.float64(0.0001)),
 (46, np.float64(0.0001)),
 (52, np.float64(0.001)),
 (52, np.float64(0.000404)),
 (52, np.float64(0.000201)),
 (52, np.float64(0.0001)),
 (53, np.float64(0.000201)),
 (53, np.float64(0.0001)),
 (54, np.float64(0.000201)),
 (54, np.float64(0.0001)),
 (55, np.float64(0.000201)),
 (55, np.float64(0.0001)),
 (56, np.float64(0.000201)),
 (56, np.float64(0.0001)),
 (57, np.float64(0.000201)),
 (57, np.float64(0.0001)),
 (58, np.float64(0.000201)),
 (58, np.float64(0.0001)),
 (59, np.float64(0.000404)),
 (59, np.float64(0.000201)),
 (59, np.float64(0.0001)),
 (23, np.float64(0.000404)),
 (23, np.float64(0.000201)),
 (23, np.float64(0.0001))]

In [11]:
for (site, poe) in sorted(prelim_ensembles_not_passing):
    e = preliminary_ensembles[(site, poe)]
    if e == None:
        continue
    failing_ims = e["ks_failed_ims"]
    print(site, poe, failing_ims)

4 0.0001 ['PGA', 'SA(0.025)', 'SA(0.033)', 'SA(0.045)', 'SA(0.059)', 'SA(0.079)', 'SA(0.106)', 'SA(0.141)', 'SA(0.188)', 'SA(0.251)', 'SA(0.335)', 'SA(0.447)', 'SA(0.597)', 'SA(0.797)', 'SA(1.063)', 'SA(1.418)', 'SA(6.0)']
4 0.000201 ['PGA', 'SA(0.025)', 'SA(0.059)', 'SA(0.079)', 'SA(0.106)', 'SA(0.188)', 'SA(0.251)', 'SA(0.335)', 'SA(0.447)', 'SA(0.597)', 'SA(0.797)', 'SA(1.063)', 'SA(1.418)', 'SA(6.0)']
4 0.000404 ['SA(0.335)', 'SA(0.447)', 'SA(0.597)', 'SA(0.797)', 'SA(1.063)', 'SA(6.0)']
4 0.001 ['SA(0.797)', 'SA(6.0)']
4 0.002103 ['SA(0.335)', 'SA(6.0)']
7 0.0001 ['RSD595']
7 0.000201 ['RSD595']
10 0.0001 ['RSD595']
10 0.000201 ['SA(6.0)']
13 0.0001 ['SA(6.0)']
13 0.000201 ['SA(6.0)']
13 0.000404 ['SA(6.0)']
14 0.0001 ['PGA', 'SA(0.025)', 'SA(0.033)', 'SA(0.045)', 'SA(0.059)', 'SA(0.079)', 'SA(0.106)', 'SA(0.141)', 'SA(0.188)', 'SA(0.251)', 'SA(0.335)', 'SA(0.447)', 'SA(0.597)', 'SA(0.797)', 'SA(1.063)', 'SA(6.0)']
14 0.000201 ['SA(0.188)', 'SA(0.251)', 'SA(0.335)', 'SA(0.447)', '

## Optimisation of Ensembles

### Round 1

Optimisation using the same parameters as the original selection

In [ ]:
only_optimise = []

optimised_selection_rd1_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_optimised_selection_rd01_rake{int(assumed_rake)}.pickle"

if LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS: 
    no_file = False
    if optimised_selection_rd1_fp.is_file():
        with open(optimised_selection_rd1_fp, "rb") as file:
            optimised_ensembles = pickle.load(file)
        print("Existing optimised ensembles loaded...")
    else:
        no_file = True
        print("No existing optimised ensembles found...")

if (not LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS) or no_file:
    # do the ensemble optimisation and save the results
    optimised_ensembles, _ = optimise_ground_motion_ensembles_for_sites(
        preliminary_ensembles, site_poe_disaggs, disagg_stats,
        gcim_dists, gm_db, basic_selection_ctx, sites, only_optimise)

    # Save the results
    with open(optimised_selection_rd1_fp, "wb") as file:
        pickle.dump(optimised_ensembles, file)

# Check if all the sites and poes found a set of suitable ground motions
optim_ensembles_not_passing_rd1 = []
for (site, poe), ensemble in optimised_ensembles.items():

    if ensemble is None:
        optim_ensembles_not_passing_rd1.append((site, poe))

    elif not ensemble["ks_passed"]:
        optim_ensembles_not_passing_rd1.append((site, poe))

if len(optim_ensembles_not_passing_rd1) == 0:
    print(f"OK! - Optimised ensembles pass for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - Optimised ensembles do not pass for {len(optim_ensembles_not_passing_rd1)} combinations of site and poe")

### Round 2

Optimisation using the same parameters as the original selection but shuffling
the selection database 3 different ways to try get different solutions.
The ensemble with the lowest score is chosen

In [ ]:
rd2_ensembles_to_optimise = [k for k in preliminary_ensembles.keys() if k in optim_ensembles_not_passing_rd1]

optimised_selection_rd2_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_optimised_selection_rd02_rake{int(assumed_rake)}.pickle"

if LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS: 
    no_file = False
    if optimised_selection_rd2_fp.is_file():
        with open(optimised_selection_rd2_fp, "rb") as file:
            optimised_ensembles = pickle.load(file)
        print("Existing optimised ensembles loaded...")
    else:
        no_file = True
        print("No existing file for Round 2 Optimisation found...")

if (not LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS) or no_file:
    # do the ensemble optimisation and save the results

    rd2_optimised_ensembles, _ = optimise_ground_motion_ensembles_for_sites_with_shuffles(
        3, [1, 2, 3], preliminary_ensembles, site_poe_disaggs, disagg_stats,
        gcim_dists, gm_db, basic_selection_ctx, sites, rd2_ensembles_to_optimise
    )

    for site_poe, new_ensemble in rd2_optimised_ensembles.items():
        optimised_ensembles[site_poe] = new_ensemble

    # Save the results
    with open(optimised_selection_rd2_fp, "wb") as file:
        pickle.dump(optimised_ensembles, file)

# Check if all the sites and poes found a set of suitable ground motions
optim_ensembles_not_passing_rd2 = []
for (site, poe), ensemble in optimised_ensembles.items():

    if ensemble is None:
        optim_ensembles_not_passing_rd2.append((site, poe))

    elif not ensemble["ks_passed"]:
        optim_ensembles_not_passing_rd2.append((site, poe))

if len(optim_ensembles_not_passing_rd2) == 0:
    print(f"OK! - Optimised ensembles pass for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - Optimised ensembles do not pass for {len(optim_ensembles_not_passing_rd2)} combinations of site and poe")

### Round 3

Optimisation with an increased number of allowable records from a single event and an increased maximum scale factor. 3 shuffles of GM Database

In [ ]:
# tweaks to the basic_selection_ctx to make the optimisation more successful
rd3_basic_selection_ctx = basic_selection_ctx.copy()
rd3_basic_selection_ctx["max_n_recs"] = 7
rd3_basic_selection_ctx["sf_bounds"] = (0.04, 30)

rd3_ensembles_to_optimise = [k for k in preliminary_ensembles.keys() if k in optim_ensembles_not_passing_rd2]

optimised_selection_rd3_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_optimised_selection_rd03_rake{int(assumed_rake)}.pickle"

if LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS: 
    no_file = False
    if optimised_selection_rd3_fp.is_file():
        with open(optimised_selection_rd3_fp, "rb") as file:
            optimised_ensembles = pickle.load(file)
        print("Existing optimised ensembles loaded...")
    else:
        no_file = True
        print("No existing file for Round 3 Optimisation found...")

if (not LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS) or no_file:
    # do the ensemble optimisation and save the results

    rd3_optimised_ensembles, _ = optimise_ground_motion_ensembles_for_sites_with_shuffles(
        3, [2, 3, 4], preliminary_ensembles, site_poe_disaggs, disagg_stats,
        gcim_dists, gm_db, rd3_basic_selection_ctx, sites, rd3_ensembles_to_optimise
    )

    for site_poe, new_ensemble in rd3_optimised_ensembles.items():
        optimised_ensembles[site_poe] = new_ensemble

    # Save the results
    with open(optimised_selection_rd3_fp, "wb") as file:
        pickle.dump(optimised_ensembles, file)

# Check if all the sites and poes found a set of suitable ground motions
optim_ensembles_not_passing_rd3 = []
for (site, poe), ensemble in optimised_ensembles.items():

    if ensemble is None:
        optim_ensembles_not_passing_rd3.append((site, poe))

    elif not ensemble["ks_passed"]:
        optim_ensembles_not_passing_rd3.append((site, poe))

if len(optim_ensembles_not_passing_rd3) == 0:
    print(f"OK! - Optimised ensembles pass for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - Optimised ensembles do not pass for {len(optim_ensembles_not_passing_rd3)} combinations of site and poe")

In [ ]:
optim_ensembles_not_passing_rd3

### Round 4

Suitable ensembles could not be found for the following sites and poes. Manual tuning of the selection parameters is required. 

In [ ]:
# tweaks to the basic_selection_ctx to make the optimisation more successful
rd4_basic_selection_ctx = basic_selection_ctx.copy()
rd4_basic_selection_ctx["max_n_recs"] = 7
rd4_basic_selection_ctx["sf_bounds"] = (0.04, 30)
# updated_basic_selection_ctx["m_bound_model"] = None
# updated_basic_selection_ctx["d_bound_model"] = None
rd4_basic_selection_ctx["vs30_bound_model"] = None

rd4_ensembles_to_optimise = [k for k in preliminary_ensembles.keys() if k in optim_ensembles_not_passing_rd3]

optimised_selection_rd4_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_optimised_selection_rd04_rake{int(assumed_rake)}.pickle"

if LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS: 
    no_file = False
    if optimised_selection_rd4_fp.is_file():
        with open(optimised_selection_rd4_fp, "rb") as file:
            optimised_ensembles = pickle.load(file)
        print("Existing optimised ensembles loaded...")
    else:
        no_file = True
        print("No existing file for Round 4 Optimisation found...")

if (not LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS) or no_file:
    # do the ensemble optimisation and save the results

    rd4_optimised_ensembles, _ = optimise_ground_motion_ensembles_for_sites_with_shuffles(
        3, [5, 6, 7], preliminary_ensembles, site_poe_disaggs, disagg_stats,
        gcim_dists, gm_db, rd4_basic_selection_ctx, sites, rd4_ensembles_to_optimise
    )

    for site_poe, new_ensemble in rd4_optimised_ensembles.items():
        optimised_ensembles[site_poe] = new_ensemble

    # Save the results
    with open(optimised_selection_rd3_fp, "wb") as file:
        pickle.dump(optimised_ensembles, file)

# Check if all the sites and poes found a set of suitable ground motions
optim_ensembles_not_passing_rd4 = []
for (site, poe), ensemble in optimised_ensembles.items():

    if ensemble is None:
        optim_ensembles_not_passing_rd4.append((site, poe))

    elif not ensemble["ks_passed"]:
        optim_ensembles_not_passing_rd4.append((site, poe))

if len(optim_ensembles_not_passing_rd4) == 0:
    print(f"OK! - Optimised ensembles pass for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - Optimised ensembles do not pass for {len(optim_ensembles_not_passing_rd4)} combinations of site and poe")

In [ ]:
optim_ensembles_not_passing_rd4

# Post-Processing

In [ ]:
# Save the best ensembles in the results folder
RESULT_FOLDER = cfg["results"]["AvgSA_06_record_selection"]

final_ensembles = {}
ensemble_list = []
ensemble_id_tuples = []
ensemble_tags = []

for (site, poe), ensemble in optimised_ensembles.items():

    if not ensemble["ks_passed"]:
        print(f"No valid ensemble: Site {site} - poe = {poe}")
        continue
    
    id_tuple = (f"site_{site}", f"{poe:g}".replace(".", "pt"))
    tag = "__".join([str(i) for i in id_tuple])
    
    # Add the gcim_quantiles from the result
    ensemble["gcim_quantiles"] = gcim_dists[(site, poe)]["stats"]

    # calculate the trt %'s
    trt_stats = pd.DataFrame(
        ensemble["recs"].groupby(("metadata", "trt")).size())
    trt_stats.columns = ["count"]
    trt_stats["proportions"] = np.round(trt_stats["count"] / \
                                        trt_stats["count"].sum(), 3)
    ensemble["trt_stats"] = trt_stats

    ensemble_list.append(ensemble)
    ensemble_id_tuples.append(id_tuple)
    ensemble_tags.append(tag)
    final_ensembles[(site, poe)] = ensemble

    with open(RESULT_FOLDER / f"{tag}__gm_selection.pickle", "wb") as file:
        pickle.dump(final_ensembles, file)
            

## Identify the number of unique records and unique events across all sites

In [ ]:
# combine all ensembles into one dataframe
rec_dfs = []
for e in final_ensembles.values():
    rec_dfs.append(e["recs"])

all_recs = pd.concat(rec_dfs, axis=0)
events = all_recs.groupby(("metadata", "event_id")).size().sort_values(ascending=False)
events = pd.DataFrame(events, columns=["count"])
events["cum_sum"] = events["count"].cumsum() 
events["cum_%"] = np.round(events["cum_sum"] / events["count"].sum(), 4)

recs_and_events = all_recs.reset_index().groupby([("metadata", "event_id"), "index"])\
    .size().sort_values(ascending=False)
recs_and_events = pd.DataFrame(recs_and_events, columns=["count"])
recs_and_events["cum_sum"] = recs_and_events["count"].cumsum() 
recs_and_events["cum_%"] = np.round(recs_and_events["cum_sum"] / recs_and_events["count"].sum(), 4)
# todo:: bar charts to show how the number of times each record and event were
# todo:: selected
recs_and_events

print(f"Number of unique events: {len(events)}")
print(f"Number of unique records: {len(recs_and_events)}")
print(f"Maximum number of times a record was selected: {recs_and_events["count"].iloc[0]} ({recs_and_events["count"].iloc[0] / len(all_recs)*100:.2f} %)")

In [ ]:
fig, ax3 = plt.subplots()
ax3.plot(recs_and_events["count"].to_numpy())
ax3.grid(True, which="both", ls="-.", color="0.8")
ax3.minorticks_on()
ax3.set_xlabel("Unique Record")
ax3.set_ylabel("Number of times selected")
ax3.set_ylim(0)
ax3.set_xlim(-10)

fig, ax1 = plt.subplots()
ax1.plot(recs_and_events["cum_%"].to_numpy())
ax1.grid(True, which="both", ls="-.", color="0.8")
ax1.minorticks_on()
ax1.set_xlabel("Unique Record")
ax1.set_ylabel("Cumulative % of all records selected")
ax1.set_ylim(0)
ax1.set_xlim(0)

fig, ax2 = plt.subplots()
ax2.plot(events["cum_%"].to_numpy())
ax2.grid(True, which="both", ls="-.", color="0.8")
ax2.minorticks_on()
ax2.set_xlabel("Unique Event")
ax2.set_ylabel("Cumulative % of all events selected")
ax2.set_ylim(0)
ax2.set_xlim(0)


### Identify the ESM and NGA Records that need to be downloaded

In [ ]:
cols_to_keep = [("metadata", "event_id"), ("metadata", "station_code"), ("metadata", "location_code")]
unique_esm_records = all_recs[all_recs[("metadata", "database")] == "ESM"]\
                     .drop_duplicates(subset=cols_to_keep)
unique_esm_records = unique_esm_records[cols_to_keep].reset_index(drop=True)
esm_records_to_download = unique_esm_records

In [ ]:
# For the NGASub records we need the NGAsubRSN for each combination of event_id and station_code
unique_NGASub_records = all_recs[all_recs[("metadata", "database")] == "NGASub"]\
                     .drop_duplicates(subset=cols_to_keep)
unique_NGASub_ids = list(unique_NGASub_records[[("metadata", "event_id"), ("metadata", "station_code")]].itertuples(index=False, name=None))

# load the NGASub database
ngasub_fp = cfg["raw_data"]["gm_flatfiles"] / "NGASub_Metadata_SA_rotD50.csv"
ngasub_db = pd.read_csv(ngasub_fp, header=0, encoding="cp1252", dtype=str)
new_index = list(ngasub_db[["NGAsubEQID", "NGAsubSSN"]].itertuples(index=False, name=None))
ngasub_db.index = new_index
ngasub_db = ngasub_db["NGAsubRSN"]

# RSNs to download
rsns_to_download = ngasub_db.loc[unique_NGASub_ids].to_list()